In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

print("🚀 Démarrage de la modélisation Gold (Schéma en Étoile & Window Functions)...")

# 1. Charger la table nettoyée depuis la couche Silver Delta Lake
silver_path = "/Volumes/workspace/default/crypto_bronze/crypto_silver_clean"
silver_df = spark.read.format("delta").load(silver_path)

# ==============================================================================
# A. DIMENSION COIN (dim_coin) - Clé de substitution technique (Surrogate Key)
# ==============================================================================
distinct_coins = silver_df.select("coin_name").distinct().sort("coin_name")
coin_window = Window.orderBy("coin_name")
dim_coin = distinct_coins.withColumn("coin_id", F.row_number().over(coin_window))

print(f"✅ Dimension dim_coin créée ({dim_coin.count()} actifs) :")
dim_coin.show(5, truncate=False)

# ==============================================================================
# B. DIMENSION DATE (dim_date) - Calendrier complet pour Time Intelligence
# ==============================================================================
distinct_dates = silver_df.select("Date").distinct().sort("Date")
dim_date = (
    distinct_dates
    .withColumn("Year", F.year("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn("Day", F.dayofmonth("Date"))
    .withColumn("DayOfWeek", F.dayofweek("Date")) # 1=Dimanche, 7=Samedi
    .withColumn("is_Weekend", F.when(F.col("DayOfWeek").isin([1, 7]), True).otherwise(False))
)

print(f"✅ Dimension dim_date créée ({dim_date.count()} dates) :")
dim_date.show(5, truncate=False)

# ==============================================================================
# C. TABLE DE FAITS (fact_marketdata) - PySpark Window Functions
# ==============================================================================
# Jointure pour injecter la clé étrangère coin_id
enriched_df = silver_df.join(dim_coin, on="coin_name", how="inner")

# Définition des spécifications de fenêtrage partitionnées par crypto et ordonnées par Date
w_coin = Window.partitionBy("coin_id").orderBy("Date")
w_7d = Window.partitionBy("coin_id").orderBy("Date").rowsBetween(-6, 0)
w_30d = Window.partitionBy("coin_id").orderBy("Date").rowsBetween(-29, 0)

# Calcul des indicateurs quantitatifs financiers
fact_marketdata = (
    enriched_df
    # Rendement journalier via lag (variation en %)
    .withColumn("lag_price", F.lag("price_usd", 1).over(w_coin))
    .withColumn(
        "daily_return_pct",
        F.when(F.col("lag_price").isNotNull() & (F.col("lag_price") > 0),
               F.round(((F.col("price_usd") - F.col("lag_price")) / F.col("lag_price")) * 100.0, 4))
        .otherwise(None)
    )
    # Moyenne mobile 7 jours sur les prix
    .withColumn("7_day_sma_price", F.round(F.avg("price_usd").over(w_7d), 6))
    # Volatilité glissante sur 30 jours (écart-type du rendement)
    .withColumn("30_day_volatility", F.round(F.stddev("daily_return_pct").over(w_30d), 4))
    # Moyenne mobile 7 jours sur les volumes
    .withColumn("7_day_sma_volume", F.round(F.avg("volume").over(w_7d), 2))
    # Sélection et ordre final des colonnes de faits
    .select(
        "coin_id",
        "Date",
        "price_usd",
        "market_cap",
        "volume",
        "daily_return_pct",
        "7_day_sma_price",
        "30_day_volatility",
        "7_day_sma_volume"
    )
)

print(f"✅ Table de faits fact_marketdata générée ({fact_marketdata.count()} lignes) :")
display(fact_marketdata.limit(10))

# ==============================================================================
# D. SAUVEGARDE DE LA COUCHE GOLD (Delta Lake)
# ==============================================================================
gold_base_path = "/Volumes/workspace/default/crypto_bronze/gold"
dim_coin.write.format("delta").mode("overwrite").save(f"{gold_base_path}/dim_coin")
dim_date.write.format("delta").mode("overwrite").save(f"{gold_base_path}/dim_date")
fact_marketdata.write.format("delta").mode("overwrite").save(f"{gold_base_path}/fact_marketdata")

print(f"🎉 Couche Gold (Schéma en Étoile) persistée avec succès en Delta Lake dans : {gold_base_path}")

🚀 Démarrage de la modélisation Gold (Schéma en Étoile & Window Functions)...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Dimension dim_coin créée (15 actifs) :
+------------+-------+
|coin_name   |coin_id|
+------------+-------+
|bitcoin     |1      |
|bitcoin-cash|2      |
|cardano     |3      |
|chainlink   |4      |
|cosmos      |5      |
+------------+-------+
only showing top 5 rows
✅ Dimension dim_date créée (365 dates) :
+----------+----+-----+---+---------+----------+
|Date      |Year|Month|Day|DayOfWeek|is_Weekend|
+----------+----+-----+---+---------+----------+
|2025-10-30|2025|10   |30 |5        |false     |
|2025-10-31|2025|10   |31 |6        |false     |
|2025-11-01|2025|11   |1  |7        |true      |
|2025-11-02|2025|11   |2  |1        |true      |
|2025-11-03|2025|11   |3  |2        |false     |
+----------+----+-----+---+---------+----------+
only showing top 5 rows
✅ Table de faits fact_marketdata générée (5475 lignes) :


coin_id,Date,price_usd,market_cap,volume,daily_return_pct,7_day_sma_price,30_day_volatility,7_day_sma_volume
1,2025-10-30,110026.08762,2.19520286098923E12,6.385739186436E10,null,110026.08762,null,6.385739186436E10
1,2025-10-31,108409.379925,2.15971913727211E12,7.274571785183E10,-1.4694,109217.733773,null,6.83015548581E10
1,2025-11-01,109566.606619,2.18495837679688E12,5.844060231349E10,1.0675,109334.024721,1.7939,6.501457067656E10
1,2025-11-02,110066.941741,2.19447277072602E12,2.524593084049E10,0.4566,109517.253976,1.324,5.507241071754E10
1,2025-11-03,110499.522383,2.20695128634651E12,3.619625887174E10,0.393,109713.707658,1.0972,5.129718034838E10
1,2025-11-04,106451.706533,2.12523494615928E12,7.495801387991E10,-3.6632,109170.040804,1.9373,5.524065260364E10
1,2025-11-05,101480.766596,2.02711737339517E12,1.1288519578055E11,-4.6697,108071.57306,2.3885,6.34755873432E10
1,2025-11-06,103878.9336,2.07242058009191E12,7.709267881422E10,2.3632,107193.4082,2.5857,6.536634262175E10
1,2025-11-07,101261.64551,2.01950561772735E12,6.40076495092E10,-2.5196,106172.303283,2.4709,6.411804714423E10
1,2025-11-08,103191.756067,2.0614834888539E12,9.353857226522E10,1.9061,105261.610347,2.5068,6.913204285162E10


🎉 Couche Gold (Schéma en Étoile) persistée avec succès en Delta Lake dans : /Volumes/workspace/default/crypto_bronze/gold
